# DWBI Week 3 — Fact & Dimension Tables
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's flat `Shipments_Unified` (34 rows, 578,650 kg) becomes **one fact table + five dimensions** — and the totals do not move by a single kilogram.

## T3 — Additivity audit

| # | Measure | Class | Safe to SUM over time? | Store instead |
|---|---|---|---|---|
| 1 | WeightKG | Additive | Yes | — |
| 2 | FreightCostPKR | Additive | Yes | — |
| 3 | ShipmentCount | Additive | Yes | — |
| 4 | TrucksAvailable | Semi-additive | No (yes across depots) | Daily snapshots; sum depots, take last across days |
| 5 | WarehouseStockKG | Semi-additive | No | Daily snapshots; report closing value |
| 6 | CostPerKG | **Non-additive** | Never | Cost + weight; `DIVIDE(SUM(cost), SUM(kg))` |
| 7 | OnTimePercent | Non-additive | No | On-time count + shipment count |
| 8 | TruckUtilisationRate | Non-additive | No | Used + available capacity |
| 9 | Days_Late | Additive | Yes (never average it) | — |
| 10 | AverageWeightPerShipment | Non-additive | No — **delete the column** | WeightKG + ShipmentCount; compute at query time |

### The proof — average of a ratio vs ratio of sums

In [1]:
import pandas as pd
u = pd.read_csv("data/Shipments_Unified.csv")
wrong = (u.CostPKR / u.WeightKG).mean()          # average the per-row ratio
right = u.CostPKR.sum() / u.WeightKG.sum()       # DIVIDE the two sums
print(f"WRONG (average of ratios): {wrong:.4f} PKR/kg")
print(f"RIGHT (ratio of sums):     {right:.4f} PKR/kg")
print(f"Gap: {(right-wrong)/wrong*100:.1f}% — nothing turns red, no error is raised.")
print("\nBoard slide gets 8.31. The 8.06 measures the average *shipment's* cost per kg")
print("(every shipment weighted equally) — a real quantity answering a different question.")

WRONG (average of ratios): 8.0604 PKR/kg
RIGHT (ratio of sums):     8.3090 PKR/kg
Gap: 3.1% — nothing turns red, no error is raised.

Board slide gets 8.31. The 8.06 measures the average *shipment's* cost per kg
(every shipment weighted equally) — a real quantity answering a different question.


**Q3** — the "delete the column" row: `AverageWeightPerShipment` (and `CostPerKG`) — never store a
pre-computed ratio; store the numerator and denominator and divide at query time.